# Gala Scout 25/26
### Galatasaray Player Scouting Pipeline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity


## 1. Load & Clean Data

In [ ]:
df = pd.read_csv('data/players_data-2025_2026.csv')
df = df[~df['Pos'].str.contains('GK', na=False)].copy()
numeric_cols = ['Age','Min','90s','Gls','Ast','G+A','G-PK','Sh','SoT','SoT%','G/Sh','G/SoT','Fls','Fld','Int','TklW','CrdY','CrdR','Mn/MP','Starts','MP']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
df = df[df['Min'] >= 500].copy()
df['Gls_90'] = df['Gls'] / df['90s']
df['Ast_90'] = df['Ast'] / df['90s']
df['GA_90']  = df['G+A'] / df['90s']
df['TklW_90'] = df['TklW'] / df['90s']
df['Int_90']  = df['Int'] / df['90s']
df['Sh_90']   = df['Sh'] / df['90s']
df['SoT_90']  = df['SoT'] / df['90s']
df['Discipline'] = df['CrdY'] + (df['CrdR'] * 3)
def classify_position(pos):
    if pd.isna(pos): return 'Unknown'
    if 'FW' in pos and 'MF' not in pos: return 'Forward'
    if 'FW' in pos and 'MF' in pos: return 'AttMid_Wing'
    if 'MF' in pos and 'DF' not in pos: return 'Midfielder'
    if 'DF' in pos: return 'Defender'
    return 'Other'
df['PosGroup'] = df['Pos'].apply(classify_position)
print(f'Players loaded: {len(df)}')


## 2. Scoring & Filters

In [ ]:
elite_clubs = ['Real Madrid','Barcelona','Manchester City','Bayern Munich','Paris Saint-Germain','Liverpool','Arsenal','Chelsea','Manchester Utd','Inter','Juventus','Atletico Madrid']
def score_players(group_df, weights):
    scored = group_df.copy()
    cols = list(weights.keys())
    scored = scored.dropna(subset=cols)
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(scored[cols])
    scaled_df = pd.DataFrame(scaled, columns=cols, index=scored.index)
    scored['ScoutScore'] = sum(scaled_df[col] * w for col, w in weights.items())
    return scored.sort_values('ScoutScore', ascending=False)
def filter_targets(scored_df, max_age=28, min_minutes=900):
    return scored_df[(scored_df['Age'] <= max_age) & (scored_df['Min'] >= min_minutes) & (~scored_df['Squad'].isin(elite_clubs))]
forward_weights   = {'Gls_90': 0.35, 'Ast_90': 0.15, 'SoT_90': 0.20, 'G/Sh': 0.20, 'G/SoT': 0.10}
midfielder_weights = {'GA_90': 0.30, 'TklW_90': 0.25, 'Int_90': 0.25, 'Ast_90': 0.20}
defender_weights  = {'TklW_90': 0.35, 'Int_90': 0.35, 'Discipline': -0.20, 'GA_90': 0.10}
attmid_weights    = {'GA_90': 0.30, 'Gls_90': 0.25, 'Ast_90': 0.25, 'SoT_90': 0.20}
fwd = score_players(df[df['PosGroup'] == 'Forward'], forward_weights)
mid = score_players(df[df['PosGroup'] == 'Midfielder'], midfielder_weights)
dfs = score_players(df[df['PosGroup'] == 'Defender'], defender_weights)
att = score_players(df[df['PosGroup'] == 'AttMid_Wing'], attmid_weights)
for name, group in [('Forwards', fwd), ('Midfielders', mid), ('Defenders', dfs), ('Att Mid/Wings', att)]:
    print(f'\n=== Top 10 {name} ===')
    print(filter_targets(group)[['Player','Squad','Comp','Age','Min','ScoutScore']].head(10).to_string(index=False))


## 3. Radar Charts

In [ ]:
def radar_chart(player_name, df, metrics, title=None):
    player = df[df['Player'] == player_name]
    if player.empty:
        print(f'{player_name} not found')
        return
    player = player.iloc[0]
    values = []
    for m in metrics:
        col = df[m].dropna()
        val = player[m]
        normalized = 0 if pd.isna(val) else (val - col.min()) / (col.max() - col.min())
        values.append(round(normalized, 3))
    values += values[:1]
    angles = [n / float(len(metrics)) * 2 * np.pi for n in range(len(metrics))]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, size=11)
    ax.set_ylim(0, 1)
    ax.plot(angles, values, linewidth=2, color='#e8112d')
    ax.fill(angles, values, alpha=0.25, color='#e8112d')
    squad = player.get('Squad', '')
    age = int(player['Age']) if not pd.isna(player['Age']) else ''
    ax.set_title(title or f"{player_name}\n{squad} | Age {age}", size=13, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f"data/{player_name.replace(' ', '_')}_radar.png", dpi=150)
    plt.show()
forward_metrics = ['Gls_90','Ast_90','SoT_90','G/Sh','G/SoT','Sh_90']
mid_metrics     = ['GA_90','TklW_90','Int_90','Ast_90','Gls_90']
def_metrics     = ['TklW_90','Int_90','Discipline']
att_metrics     = ['Gls_90','Ast_90','GA_90','SoT_90','Sh_90']
for p in ['Donyell Malen','Jonathan Burkardt','Nikola Krstović']:
    radar_chart(p, df, forward_metrics)
for p in ['Lamine Camara','Sergi Altimira','Mamadou Sangare']:
    radar_chart(p, df, mid_metrics)
for p in ['Nnamdi Collins','Kassoum Ouattara','Malang Sarr']:
    radar_chart(p, df, def_metrics)
for p in ['Endrick','Karim Adeyemi','Ansu Fati']:
    radar_chart(p, df, att_metrics)


## 4. Similarity Model

In [ ]:
def find_similar_players(target_name, df, metrics, top_n=10, max_age=28, min_minutes=900):
    pool = df[(df['Age'] <= max_age) & (df['Min'] >= min_minutes) & (~df['Squad'].isin(elite_clubs))].copy()
    pool = pool.dropna(subset=metrics).reset_index(drop=True)
    if target_name not in pool['Player'].values:
        target_row = df[df['Player'] == target_name].dropna(subset=metrics)
        if target_row.empty:
            print(f'{target_name} not found')
            return None
        pool = pd.concat([target_row, pool], ignore_index=True)
    scaler = StandardScaler()
    scaled = scaler.fit_transform(pool[metrics])
    target_idx = pool[pool['Player'] == target_name].index[0]
    similarities = cosine_similarity([scaled[target_idx]], scaled)[0]
    pool['Similarity'] = similarities
    results = pool[pool['Player'] != target_name].sort_values('Similarity', ascending=False)
    return results[['Player','Squad','Comp','Age','Min','Similarity'] + metrics].head(top_n)
print('=== Similar to Donyell Malen ===')
print(find_similar_players('Donyell Malen', df, forward_metrics).to_string(index=False))
print('\n=== Similar to Lamine Camara ===')
print(find_similar_players('Lamine Camara', df, mid_metrics).to_string(index=False))
print('\n=== Similar to Nnamdi Collins ===')
print(find_similar_players('Nnamdi Collins', df, def_metrics).to_string(index=False))
print('\n=== Similar to Karim Adeyemi ===')
print(find_similar_players('Karim Adeyemi', df, att_metrics).to_string(index=False))
